# Baseline 2: standard GRPO

Trains a LoRA adapter on Qwen3.5-0.8B with GRPO and no skills, then measures it on the MATH dev set with no skills in the prompt. It evaluates the untrained model and each checkpoint on a fixed 200-problem dev subset for the learning curve, and the final adapter on the full 1,000-problem dev set.

Choose `MAX_STEPS` before the first run, because the learning-rate schedule depends on it. As a rough guide, use 100 on an A100 or L4 and 60 on a T4.

Setup, needed once per account:

1. Select a GPU in **Runtime > Change runtime type**. An A100 or L4 is much faster than a T4.
2. Add a read-only GitHub token for this repository in Colab's **Secrets** panel (the key icon) as `GH_TOKEN`, and turn on notebook access.

Then choose **Runtime > Run all** and check every box on the Google Drive permission popup. Keep this tab open while it runs. If the session disconnects, choose **Run all** again and it continues from where it stopped.

In [ ]:
# Settings. Choose these before the first run and do not change them afterward.
USE_GOOGLE_DRIVE = True  # False keeps results on the runtime's disk, erased when the session ends.
RUN_NAME = "grpo-baseline-seed230"
MAX_STEPS = 100  # Must be a multiple of 20, the checkpoint interval.
EVAL_BATCH_SIZE = 32

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU found. Select one in Runtime > Change runtime type, then run all.")
get_ipython().system("nvidia-smi --query-gpu=name,memory.total --format=csv")

if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/self-derived-skills/runs")
else:
    RUN_ROOT = Path("/content/runs")
RUN_DIR = RUN_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved to {RUN_DIR}")

REPO_URL = "https://github.com/Ruichu-Xia/self-derived-skills.git"
REPO_DIR = "/content/self-derived-skills"

# The token stays in an environment variable so that it never appears in a command, URL, or error.
GIT_ENV = {**os.environ, "GH_TOKEN": userdata.get("GH_TOKEN")}
CREDENTIAL_HELPER = (
    "credential.helper=!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
)


def git(*args, cwd=None):
    """Run a git command that authenticates with the token in the environment."""
    subprocess.run(["git", "-c", CREDENTIAL_HELPER, *args], env=GIT_ENV, cwd=cwd, check=True)


if os.path.isdir(REPO_DIR):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
get_ipython().system("git log -1 --format='%h %s'")

print("Installing packages. This takes a few minutes.")
install = subprocess.run(["bash", "scripts/colab_install.sh"], capture_output=True, text=True)
print(install.stdout[-1500:])
if install.returncode != 0:
    print(install.stderr[-4000:])
    raise RuntimeError("Installing the packages failed. See the output above.")

In [ ]:
def evaluate(split, name, adapter=None):
    """Run one evaluation, skipping finished work, and stop the notebook if it fails."""
    output_dir = RUN_DIR / name
    adapter_flag = f'--adapter "{adapter}"' if adapter else ""
    get_ipython().system(
        f"python experiments/evaluate_checkpoint.py --config configs/grpo_math.yaml "
        f'--output-dir "{output_dir}" --split {split} --batch-size {EVAL_BATCH_SIZE} {adapter_flag}'
    )
    metrics_path = output_dir / "metrics.json"
    if not metrics_path.exists():
        raise RuntimeError(f"Evaluation {name} failed. See the traceback above.")
    return json.loads(metrics_path.read_text())

## Train

In [ ]:
get_ipython().system(
    "python experiments/train_grpo.py --config configs/grpo_math.yaml "
    f'--output-dir "{RUN_DIR}" --max-steps {MAX_STEPS}'
)
if not (RUN_DIR / "final_adapter").exists():
    raise RuntimeError(
        "Training did not finish. Choose Run all to resume from the last checkpoint."
    )

## Evaluate

In [ ]:
checkpoint_dir = RUN_DIR / "checkpoints"
steps = sorted(int(path.name.split("-")[1]) for path in checkpoint_dir.glob("checkpoint-*"))

evaluate("dev-subset", "evals/step-0")
for step in steps:
    evaluate("dev-subset", f"evals/step-{step}", checkpoint_dir / f"checkpoint-{step}")

final_dev = evaluate("dev", "evals/final-dev", RUN_DIR / "final_adapter")

## Results

In [ ]:
get_ipython().system(f'python experiments/summarize_grpo.py --run-dir "{RUN_DIR}"')

learning_curve = json.loads((RUN_DIR / "learning_curve.json").read_text())
print(f"Final adapter on the full dev set: {final_dev['accuracy']:.3f} (n={final_dev['n']})")
print(
    json.dumps({"learning_curve": learning_curve, "final_dev": final_dev}, indent=2, sort_keys=True)
)